# 🛡️ DeepShield AI — Model Training

Fine-tunes **MobileNetV3-Small** (ImageNet-pretrained) to classify face images as **real** or **fake (deepfake)**, using the *140k Real and Fake Faces* dataset (70k real faces from Flickr + 70k StyleGAN-generated faces).

**Output:** `deepshield_mobilenetv3.pth` (~6 MB) — drop it into your project's `models/` folder.

---

### ⚠️ Before running — turn on the GPU
**Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**

Then: **Runtime → Run all**. Total time ≈ 20–35 minutes (download + training).

In [ ]:
# ── 1. Environment check ─────────────────────────────────────────
import torch
print('PyTorch :', torch.__version__)
print('GPU     :', torch.cuda.is_available() and torch.cuda.get_device_name(0) or 'NONE')
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then Run all again.'
device = torch.device('cuda')

In [ ]:
# ── 2. Download the dataset (~4 GB, a few minutes) ───────────────
import kagglehub, os

path = kagglehub.dataset_download('xhlulu/140k-real-and-fake-faces')
print('Downloaded to:', path)

# Locate the folder that contains train/valid/test
DATA = None
for root, dirs, _ in os.walk(path):
    if {'train', 'valid', 'test'} <= set(dirs):
        DATA = root
        break
assert DATA, 'train/valid/test folders not found in the dataset'
print('Data root    :', DATA)
print('Class folders:', sorted(os.listdir(os.path.join(DATA, 'train'))))

> **If the download errors with an auth message:** create a free Kaggle account → *Settings → API → Create New Token* (downloads `kaggle.json`) → run `import kagglehub; kagglehub.login()` in a new cell and paste the credentials, then re-run the cell above. Public datasets usually need **no** login.

In [ ]:
# ── 3. Configuration (V2-Heavy: full data, anti-shortcut) ────────
# Budget: ~2.5h on a T4 — fits inside a 3h Colab session with margin.
SUBSET_PER_CLASS = 50000  # FULL training set (50k per class)
VAL_PER_CLASS    = 2500   # validation images per class
EPOCHS           = 10
BATCH            = 128
LR               = 3e-4
IMG              = 224    # MobileNetV3 input size — matches the app spec
SEED             = 42

import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ── 4. Datasets & loaders ────────────────────────────────────────
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet

# ---- Anti-shortcut augmentation (V2) ----
# Without these, the model memorizes the dataset's resize/JPEG pipeline
# fingerprint instead of real GAN artifacts — 97% on this test set but
# blind to StyleGAN faces processed any other way (we learned this the
# hard way). Randomizing compression/resolution forces it to learn the
# actual generative artifacts.
import io
from PIL import Image as PILImage

class RandomJPEG:
    """Re-encode at a random JPEG quality — kills compression fingerprints."""
    def __init__(self, p=0.7, quality=(30, 95)):
        self.p, self.quality = p, quality
    def __call__(self, img):
        if random.random() < self.p:
            buf = io.BytesIO()
            img.save(buf, 'JPEG', quality=random.randint(*self.quality))
            buf.seek(0)
            img = PILImage.open(buf).convert('RGB')
        return img

class RandomRescale:
    """Down-then-up scaling — kills resolution fingerprints."""
    def __init__(self, p=0.5, lo=0.5):
        self.p, self.lo = p, lo
    def __call__(self, img):
        if random.random() < self.p:
            w, h = img.size
            s = random.uniform(self.lo, 1.0)
            img = img.resize((max(32, int(w*s)), max(32, int(h*s))), PILImage.BILINEAR)
            img = img.resize((w, h), PILImage.BILINEAR)
        return img

class FixedJPEG:
    """Deterministic q40 re-encode — used ONLY to build the 'robust'
    validation set that measures shortcut-resistance every epoch."""
    def __init__(self, quality=40):
        self.q = quality
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=self.q)
        buf.seek(0)
        return PILImage.open(buf).convert('RGB')

train_tf = transforms.Compose([
    RandomRescale(),
    RandomJPEG(),
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
robust_tf = transforms.Compose([
    FixedJPEG(40),
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_train  = datasets.ImageFolder(f'{DATA}/train', train_tf)
full_valid  = datasets.ImageFolder(f'{DATA}/valid', eval_tf)
full_robust = datasets.ImageFolder(f'{DATA}/valid', robust_tf)  # same imgs, corrupted
full_test   = datasets.ImageFolder(f'{DATA}/test',  eval_tf)
CLASSES = full_train.classes  # alphabetical: ['fake', 'real']
print('Classes:', CLASSES)

def subset_per_class(ds, per_class, seed=SEED):
    """Balanced random subset (deterministic, so valid == robust images)."""
    rng = random.Random(seed)
    by_class = {}
    for idx, (_, y) in enumerate(ds.samples):
        by_class.setdefault(y, []).append(idx)
    keep = []
    for idxs in by_class.values():
        rng.shuffle(idxs)
        keep += idxs[:per_class]
    rng.shuffle(keep)
    return Subset(ds, keep)

train_ds  = subset_per_class(full_train,  SUBSET_PER_CLASS)
valid_ds  = subset_per_class(full_valid,  VAL_PER_CLASS)
robust_ds = subset_per_class(full_robust, VAL_PER_CLASS)  # same seed → same images

train_dl  = DataLoader(train_ds,  batch_size=BATCH, shuffle=True,  num_workers=4,
                       pin_memory=True, persistent_workers=True)
valid_dl  = DataLoader(valid_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
robust_dl = DataLoader(robust_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl   = DataLoader(full_test, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train {len(train_ds)} · Valid {len(valid_ds)} (+robust copy) · Test {len(full_test)}')

In [ ]:
# ── 5. Peek at the data ──────────────────────────────────────────
import matplotlib.pyplot as plt

def denorm(t):
    return (t.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)

xb, yb = next(iter(train_dl))
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, img, y in zip(axes.flat, xb, yb):
    ax.imshow(denorm(img)); ax.set_title(CLASSES[y]); ax.axis('off')
plt.suptitle('Training samples'); plt.show()

In [ ]:
# ── 6. Model: MobileNetV3-Small, 2-class head ────────────────────
from torchvision import models

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, len(CLASSES))
model = model.to(device)

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'MobileNetV3-Small ready — {params:.1f}M parameters')

In [ ]:
# ── 7. Train (V2-Heavy: cosine schedule, robust-checkpoint selection) ──
import copy
from tqdm.auto import tqdm

# Label smoothing: stops the model from becoming overconfident on
# dataset-specific quirks — helps generalization.
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(train_dl), eta_min=1e-5)

@torch.no_grad()
def evaluate(dl):
    model.eval()
    correct = total = 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        correct += (model(xb).argmax(1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

history = {'loss': [], 'val_acc': [], 'robust_acc': []}
best_robust, best_val, best_state = 0.0, 0.0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    bar = tqdm(train_dl, desc=f'Epoch {epoch}/{EPOCHS}')
    for xb, yb in bar:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running += loss.item() * yb.size(0)
        bar.set_postfix(loss=f'{loss.item():.3f}',
                        lr=f"{optimizer.param_groups[0]['lr']:.1e}")

    epoch_loss = running / len(train_ds)
    val_acc    = evaluate(valid_dl)
    robust_acc = evaluate(robust_dl)   # ← same images, JPEG-q40 corrupted
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['robust_acc'].append(robust_acc)
    print(f'Epoch {epoch}: loss {epoch_loss:.4f} · val {val_acc*100:.2f}% '
          f'· robust {robust_acc*100:.2f}%')

    # Select the best checkpoint by ROBUST accuracy — the whole point of
    # V2 is surviving images the training pipeline never produced.
    if robust_acc > best_robust:
        best_robust, best_val = robust_acc, val_acc
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, 'best_so_far.pth')  # crash insurance —
        # if the session dies mid-run, grab this from the Files panel.

model.load_state_dict(best_state)
print(f'\nBest: robust {best_robust*100:.2f}% · val {best_val*100:.2f}%')

In [ ]:
# ── 8. Training curves (save these for your report!) ─────────────
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
ep = range(1, len(history['loss']) + 1)
a1.plot(ep, history['loss'], marker='o')
a1.set_title('Training loss'); a1.set_xlabel('epoch')
a2.plot(ep, [a * 100 for a in history['val_acc']], marker='o',
        color='green', label='clean val')
a2.plot(ep, [a * 100 for a in history['robust_acc']], marker='s',
        color='orange', label='robust val (JPEG q40)')
a2.set_title('Validation accuracy (%)'); a2.set_xlabel('epoch'); a2.legend()
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

# The gap between the two lines = how much the model still leans on
# pipeline shortcuts. Small gap = robust model. Report gold.

In [ ]:
# ── 9. Final evaluation on the untouched TEST set ────────────────
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_dl, desc='Testing'):
        all_preds += model(xb.to(device)).argmax(1).cpu().tolist()
        all_true  += yb.tolist()

test_acc = (np.array(all_preds) == np.array(all_true)).mean()
print(f'TEST accuracy: {test_acc*100:.2f}%\n')
print(classification_report(all_true, all_preds, target_names=CLASSES))

cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1], CLASSES); ax.set_yticks([0, 1], CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix')
plt.savefig('confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# ── 10. Sample predictions (viva material) ───────────────────────
xb, yb = next(iter(test_dl))
with torch.no_grad():
    probs = torch.softmax(model(xb[:12].to(device)), dim=1).cpu()

fig, axes = plt.subplots(2, 6, figsize=(14, 5.5))
for ax, img, true_y, p in zip(axes.flat, xb, yb, probs):
    pred = p.argmax().item()
    ax.imshow(denorm(img))
    ok = pred == true_y.item()
    ax.set_title(f'{CLASSES[pred]} {p[pred]*100:.0f}%',
                 color='green' if ok else 'red', fontsize=10)
    ax.axis('off')
plt.suptitle('Model predictions (green = correct)'); plt.show()

In [ ]:
# ── 11. Export the checkpoint for DeepShield ─────────────────────
CKPT = 'deepshield_mobilenetv3.pth'
torch.save({
    'arch':          'mobilenet_v3_small',
    'state_dict':    model.state_dict(),
    'classes':       CLASSES,          # ['fake', 'real'] — index order matters!
    'input_size':    IMG,
    'normalize':     {'mean': MEAN, 'std': STD},
    'val_accuracy':  round(best_val * 100, 2),
    'robust_val_accuracy': round(best_robust * 100, 2),
    'test_accuracy': round(float(test_acc) * 100, 2),
    'trained_on':    f'140k-real-and-fake-faces (V2-Heavy: {SUBSET_PER_CLASS}/class, '
                     f'{EPOCHS} epochs, anti-shortcut aug, robust-selected)',
}, CKPT)

import os
print(f'Saved {CKPT} — {os.path.getsize(CKPT)/1e6:.1f} MB')

from google.colab import files
files.download(CKPT)  # downloads to your computer

In [ ]:
---
## ✅ Done — next steps

1. **`deepshield_mobilenetv3.pth`** downloaded → replace the old file in **`g:\deepfake\models\`** (the backend picks it up automatically, no restart needed).
2. **`demo_samples.zip`** downloaded → unzip somewhere handy (e.g. `g:\deepfake\training\demo_samples\`). These 24 images are your **live-demo ammunition**: the model has never seen them, filenames reveal nothing, and `ANSWER_KEY.txt` has the ground truth. Expect ~97% of them to be called correctly.
3. Also grab `training_curves.png` + `confusion_matrix.png` from the Files panel for the report.

> **Honest limitations for the report/viva (write these — they show maturity):**
> - Trained on **StyleGAN** fakes (140k dataset). Newer generators (diffusion models — Midjourney/SD-era faces) are out of scope.
> - V2 augmentation (random JPEG / rescale) reduces *pipeline shortcut learning*, improving robustness to differently-processed images — but cross-generator generalization is a genuinely open research problem.
> - Designed for **face portraits**; verdicts on non-face images are not meaningful.

---
## ✅ Done — next steps

1. The file **`deepshield_mobilenetv3.pth`** downloaded to your computer (also grab `training_curves.png` + `confusion_matrix.png` from the Files panel on the left — great for your project report).
2. Move the `.pth` file into your project: **`g:\deepfake\models\`**
3. Back in the project, run **Phase 4** — the Flask backend will load this checkpoint and replace the simulated verdicts with real inference.

> **Note for the report/viva:** this model is trained on StyleGAN-generated faces. It detects that family of fakes well; newer generators (diffusion models) may fool it — that limitation is worth one honest line in your report.